In [2]:
!pip install requests

  Using cached requests-2.33.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
Using cached requests-2.33.1-py3-none-any.whl (64 kB)
Using cached urllib3-2.6.3-py3-none-any.whl (131 kB)
Using cached certifi-2026.2.25-py3-none-any.whl (153 kB)

   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   -------- ------------------------------- 1/5 [idna]
   -------- ------------------------------- 1/5 [idna]
   ---------------- ----------------------- 2/5 [charset_normalizer]
   ---------------- ----------------------- 2/5 [charset_normalizer]
   ---------------- ---------------------

In [3]:
# Movie Recommender - Advanced Version

import numpy as np
import pandas as pd
import ast
import requests
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle
import nltk
from nltk.stem.porter import PorterStemmer

nltk.download('punkt')

[nltk_data] Downloading package punkt to C:\Users\Aarav
[nltk_data]     Mah\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [4]:
# -----------------------------
# 1. Load Data
# -----------------------------
credits = pd.read_csv('data/tmdb_5000_credits.csv')
movies = pd.read_csv('data/tmdb_5000_movies.csv')

In [5]:
# Merge on title
movies = movies.merge(credits, on='title')

In [6]:
# Keep relevant columns
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]

In [7]:
# Drop nulls & duplicates
movies.dropna(inplace=True)
movies.drop_duplicates(inplace=True)

# -----------------------------
# 2. Preprocess Data
# -----------------------------

# Convert stringified lists to python lists
def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

def convert3(obj):
    L=[]
    counter=0
    for i in ast.literal_eval(obj):
        if counter !=3:
            L.append(i['name'])
            counter +=1
        else:
            break
    return L

def fetch_director(obj):
    L=[]
    for i in ast.literal_eval(obj):
        if i['job']=='Director':
            L.append(i['name'])
            break
    return L

movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies['cast'] = movies['cast'].apply(convert3)
movies['crew'] = movies['crew'].apply(fetch_director)
movies['overview'] = movies['overview'].apply(lambda x: x.split())

In [9]:
# Remove spaces
for col in ['genres','keywords','cast','crew']:
    movies[col] = movies[col].apply(lambda x: [i.replace(" ","") for i in x])

In [10]:
# Combine all text features
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [11]:
# Create final dataframe
new_df = movies[['movie_id','title','tags']]
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

C:\Users\Aarav Mah\AppData\Local\Temp\ipykernel_5264\2775934394.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))
C:\Users\Aarav Mah\AppData\Local\Temp\ipykernel_5264\2775934394.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())


In [12]:
# -----------------------------
# 3. Text Vectorization
# -----------------------------
# TF-IDF vectorizer
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
vectors = tfidf.fit_transform(new_df['tags']).toarray()

In [14]:
# -----------------------------
# 4. Stemming
# -----------------------------
ps = PorterStemmer()
def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

new_df.loc[:,'tags'] = new_df['tags'].apply(stem)

In [15]:
# Recompute vectors after stemming
vectors = tfidf.fit_transform(new_df['tags']).toarray()

In [16]:
# -----------------------------
# 5. Cosine Similarity
# -----------------------------
similarity = cosine_similarity(vectors)

In [ ]:
import requests

OMDB_API_KEY = "8d60250e"  

def get_movie_details_omdb(title):
    url = f"http://www.omdbapi.com/?t={title}&apikey={OMDB_API_KEY}"
    response = requests.get(url).json()
    if response['Response'] == "True":
        return {
            "title": response['Title'],
            "year": response['Year'],
            "director": response['Director'],
            "actors": response['Actors'],
            "plot": response['Plot'],
            "poster": response['Poster'],
            "imdbRating": response['imdbRating']
        }
    else:
        return None

def recommend(movie):
    if movie not in new_df['title'].values:
        print("Movie not in dataset. Try similar spelling or a different movie.")
        return
    
    movie_index = new_df[new_df['title'] == movie].index[0]
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x:x[1])[1:6]
    
    print(f"\nTop 5 Recommendations for '{movie}':\n")
    
    for i in movies_list:
        title = new_df.iloc[i[0]].title
        details = get_movie_details_omdb(title)
        if details:
            print(f"{details['title']} ({details['year']}) -> IMDB: {details['imdbRating']}")
            print(f"Plot: {details['plot']}")
            print(f"Poster URL: {details['poster']}\n")
        else:
            print(title)

In [20]:
# -----------------------------
# 8. Save Model & Data
# -----------------------------
pickle.dump(new_df.to_dict(), open('models/movie_dict.pkl','wb'))
pickle.dump(similarity, open('models/similarity.pkl','wb'))

print("Model and data saved!")

Model and data saved!
